# Vector Databases — How They Work and Why They're Different

**Goal:** Understand the fundamental differences between traditional databases and vector databases, see how similarity search works under the hood, and learn why retrieval quality is the make-or-break factor in RAG systems.

**No API keys required** — this notebook uses open-source models that run locally.

---

## What You'll Learn

1. How vector databases differ from traditional SQL databases (mental model + code)
2. The full pipeline: text → embedding → vector → stored with metadata
3. How cosine similarity, Euclidean distance, and dot product compare
4. Why vector search is non-deterministic (and what that means for your system)
5. What happens when retrieval goes wrong — the "wrong context" problem
6. Practical techniques to improve retrieval quality

## Setup

Install dependencies if you haven't already:

In [ ]:
# Uncomment and run if you need to install dependencies
# !pip install sentence-transformers scikit-learn matplotlib plotly numpy pandas

In [ ]:
# === Standard library imports ===
import copy
import textwrap

# === Data & math ===
import numpy as np
import pandas as pd

# === Visualization ===
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === ML / embeddings ===
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

import warnings
warnings.filterwarnings('ignore')

# Load embedding model (downloads ~80MB on first run)
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✅ Model loaded: all-MiniLM-L6-v2 ({model.get_sentence_embedding_dimension()} dimensions)")
print(f"\nSetup complete!")

## Step 1: Traditional DB vs Vector DB — The Mental Model

**Traditional (SQL) databases** store structured rows and columns. You query with exact conditions:
```sql
SELECT * FROM reports WHERE revenue > 100 AND quarter = 'Q3'
```
This is precise, deterministic, and fast — but it only works when you know *exactly* what to look for.

**Vector databases** store high-dimensional vectors (embeddings) alongside metadata. You query by *similarity*:
```
"Find me documents similar to: 'How is the company doing financially?'"
```
This finds semantically related results even when the exact words don't match — but the results are approximate and score-based.

Let's see both approaches side by side.

In [ ]:
# ============================================================
# PART A: Traditional Database — Exact Match Query
# ============================================================

# Simulate a traditional SQL table with structured CorpX data
corpx_table = pd.DataFrame({
    'doc_id':   [1, 2, 3, 4, 5, 6, 7, 8],
    'category': ['financial', 'financial', 'hr', 'hr', 'client', 'client', 'tech', 'tech'],
    'quarter':  ['Q3', 'Q2', 'Q3', 'Q3', 'Q3', 'Q2', 'Q3', 'Q3'],
    'metric':   ['revenue', 'revenue', 'headcount', 'attrition', 'nps', 'nps', 'uptime', 'ai_queries'],
    'value':    [38.4, 33.7, 620, 9.0, 58, 54, 99.7, 280],
    'text': [
        'Q3 revenue reached $38.4M, up 14% year-over-year.',
        'Q2 revenue was $33.7M, up 11% year-over-year.',
        'Total headcount reached 620 employees, net increase of 22.',
        'Voluntary attrition fell to 9% annualized from 12% last year.',
        'Net Promoter Score improved to 58, up from 54 last quarter.',
        'Q2 Net Promoter Score was 54, steady from prior quarter.',
        'Platform uptime maintained at 99.7% across all services.',
        'The GenAI dashboard processes 280 queries per day.',
    ]
})

# SQL-style query: SELECT * FROM corpx WHERE category = 'financial' AND quarter = 'Q3'
sql_result = corpx_table[
    (corpx_table['category'] == 'financial') & (corpx_table['quarter'] == 'Q3')
]

print("🔍 TRADITIONAL DB QUERY")
print("   SQL: SELECT * FROM corpx WHERE category='financial' AND quarter='Q3'")
print(f"   Results: {len(sql_result)} row(s)\n")
print(sql_result[['doc_id', 'category', 'quarter', 'text']].to_string(index=False))

print("\n" + "=" * 80)

# ============================================================
# PART B: Vector Database — Similarity Query
# ============================================================

# Embed all documents in the table
all_texts = corpx_table['text'].tolist()
all_embeddings = model.encode(all_texts)

# Natural language query — no need for exact column names or values
query = "How is the company doing financially this quarter?"
query_embedding = model.encode([query])

# Compute cosine similarity between query and every document
similarities = cosine_similarity(query_embedding, all_embeddings)[0]

# Rank results by similarity score (highest first)
ranked_idx = np.argsort(similarities)[::-1]

print("\n🔍 VECTOR DB QUERY")
print(f'   Query: "{query}"')
print(f"   Results ranked by similarity:\n")
for rank, idx in enumerate(ranked_idx, 1):
    row = corpx_table.iloc[idx]
    # Show similarity score and the document text
    marker = "✅" if similarities[idx] > 0.4 else "  "
    print(f"   {marker} {rank}. (sim: {similarities[idx]:.4f}) [{row['category']}] {row['text']}")

print("\n💡 Key difference: The SQL query requires you to know the schema (category='financial').")
print("   The vector query understands natural language — 'doing financially' matches 'revenue'.")
print("   But notice: vector results are RANKED, not filtered. Every document gets a score.")

In [ ]:
# ============================================================
# Visualization: Side-by-side comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left panel: Traditional DB (binary match) ---
# In SQL, results are either IN (1) or OUT (0) — no in-between
sql_matches = [
    1 if (corpx_table.iloc[i]['category'] == 'financial' and corpx_table.iloc[i]['quarter'] == 'Q3')
    else 0
    for i in range(len(corpx_table))
]
labels = [f"Doc {i+1}: {corpx_table.iloc[i]['text'][:35]}..." for i in range(len(corpx_table))]

colors_sql = ['#27AE60' if m == 1 else '#E0E0E0' for m in sql_matches]
axes[0].barh(range(len(labels)), sql_matches, color=colors_sql, edgecolor='white')
axes[0].set_yticks(range(len(labels)))
axes[0].set_yticklabels(labels, fontsize=8)
axes[0].set_xlim(0, 1.1)
axes[0].set_xlabel('Match (0 = No, 1 = Yes)')
axes[0].set_title('Traditional DB: Binary Match\n(category=financial AND quarter=Q3)',
                   fontsize=11, fontweight='bold')
axes[0].invert_yaxis()

# --- Right panel: Vector DB (continuous similarity) ---
# Every document gets a similarity score — a continuous spectrum
colors_vec = ['#2E86C1' if s > 0.4 else '#B0C4DE' for s in similarities]
axes[1].barh(range(len(labels)), similarities, color=colors_vec, edgecolor='white')
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels(labels, fontsize=8)
axes[1].set_xlim(0, 1.0)
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_title('Vector DB: Continuous Similarity\n("How is the company doing financially?")',
                   fontsize=11, fontweight='bold')
axes[1].invert_yaxis()

# Add a vertical threshold line on the vector panel
axes[1].axvline(x=0.4, color='red', linestyle='--', alpha=0.7, label='Threshold (0.4)')
axes[1].legend(fontsize=9)

plt.suptitle('Traditional DB vs Vector DB: How Results Are Selected',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Left: SQL gives you a hard YES/NO. Right: Vector DB gives a similarity spectrum.")
print("   The red dashed line is a threshold — you choose where to draw the cutoff.")
print("   This threshold choice is one reason vector search is non-deterministic in practice.")

## Step 2: How Data Gets Into a Vector DB

The ingestion pipeline for a vector database looks like this:

```
Raw Text  →  Embedding Model  →  Vector (384 dims)  →  Stored with Metadata
```

Each document chunk becomes a high-dimensional vector, and you store it alongside metadata (source, category, date) so you can filter later.

Let's build this pipeline with realistic CorpX data.

In [ ]:
# ============================================================
# CorpX document chunks — a realistic mix of enterprise content
# These simulate what you'd get after chunking real documents.
# ============================================================

corpx_documents = [
    # --- Financial Reports ---
    {"text": "Q3 2024 revenue reached $38.4 million, representing a 14% increase year-over-year. Growth was driven primarily by expansion in federal technology contracts.",
     "source": "Q3 2024 Earnings Report", "category": "financial", "date": "2024-10-15"},
    {"text": "Gross margin improved to 37.2% from 35.1% in Q2, reflecting better project mix and improved utilization rates across consulting teams.",
     "source": "Q3 2024 Earnings Report", "category": "financial", "date": "2024-10-15"},
    {"text": "Operating expenses held flat at $10.8 million despite 14% revenue growth, demonstrating strong cost discipline and operational leverage.",
     "source": "Q3 2024 Earnings Report", "category": "financial", "date": "2024-10-15"},
    {"text": "Q2 2024 revenue was $33.7 million, up 11% year-over-year. The pipeline remains strong with $98M in qualified opportunities.",
     "source": "Q2 2024 Earnings Report", "category": "financial", "date": "2024-07-12"},

    # --- HR & People Updates ---
    {"text": "Total headcount reached 620 employees in Q3, a net increase of 22. The distributed workforce now spans 14 states and 3 time zones.",
     "source": "Q3 People Update", "category": "hr", "date": "2024-10-20"},
    {"text": "Voluntary attrition fell to 9% annualized, down from 12% last year. Exit interviews cite improved career development programs.",
     "source": "Q3 People Update", "category": "hr", "date": "2024-10-20"},
    {"text": "The firm hired 32 new consultants in Q3 with an 87% offer acceptance rate. Average time-to-fill dropped to 28 days from 35.",
     "source": "Q3 People Update", "category": "hr", "date": "2024-10-20"},
    {"text": "Employee engagement survey results: overall score 4.1/5.0, with highest marks in team collaboration (4.4) and lowest in compensation satisfaction (3.6).",
     "source": "Annual Engagement Survey", "category": "hr", "date": "2024-09-01"},

    # --- Client Notes ---
    {"text": "Client satisfaction averaged 4.5 out of 5.0 in Q3, a firm record. Federal clients rated highest at 4.7, commercial at 4.3.",
     "source": "Q3 Client Report", "category": "client", "date": "2024-10-18"},
    {"text": "Net Promoter Score improved to 58 in Q3, up from 54 in Q2 and 49 a year ago. Six existing clients expanded their engagements.",
     "source": "Q3 Client Report", "category": "client", "date": "2024-10-18"},
    {"text": "The firm onboarded 10 new clients in Q3, bringing total active clients to 78. New client acquisition cost decreased by 15%.",
     "source": "Q3 Client Report", "category": "client", "date": "2024-10-18"},
    {"text": "MegaRetail engagement expanded from 2 to 5 workstreams. They are now our third-largest commercial client by revenue.",
     "source": "Client Expansion Notes", "category": "client", "date": "2024-10-05"},

    # --- Technology Updates ---
    {"text": "Azure cloud migration is 85% complete across all business units. Remaining 15% involves legacy applications scheduled for Q4 migration.",
     "source": "Tech Modernization Update", "category": "tech", "date": "2024-10-22"},
    {"text": "The GenAI dashboard processes 280 queries per day, up from 150 in Q2. User adoption reached 68% of senior leadership.",
     "source": "AI Platform Report", "category": "tech", "date": "2024-10-22"},
    {"text": "FedRAMP authorization maintained with zero security incidents. SOC 2 Type II audit completed with no findings.",
     "source": "Security Compliance Report", "category": "tech", "date": "2024-10-25"},
    {"text": "Cloud infrastructure costs are tracking 15% below budget due to reserved instance optimization and auto-scaling improvements.",
     "source": "Tech Modernization Update", "category": "tech", "date": "2024-10-22"},

    # --- Strategy Memos ---
    {"text": "The board approved a new strategic focus on grid modernization and clean energy advisory services, targeting $12M in new revenue by 2025.",
     "source": "Strategy Memo", "category": "strategy", "date": "2024-09-30"},
    {"text": "AI-focused engagements now represent 15% of total revenue, up from 7% a year ago. The goal is 25% by end of 2025.",
     "source": "Strategy Memo", "category": "strategy", "date": "2024-09-30"},
]

print(f"✅ Created {len(corpx_documents)} CorpX document chunks")
print(f"\n   Category breakdown:")
for cat in ['financial', 'hr', 'client', 'tech', 'strategy']:
    count = sum(1 for d in corpx_documents if d['category'] == cat)
    print(f"   - {cat}: {count} chunks")

In [ ]:
# ============================================================
# The full ingestion pipeline: Text → Embedding → Vector Store
# ============================================================

# Step 1: Extract just the text from each document chunk
texts = [doc["text"] for doc in corpx_documents]

# Step 2: Run all texts through the embedding model
# This converts each text string into a 384-dimensional vector
embeddings = model.encode(texts)

print(f"🔍 Embedding pipeline results:")
print(f"   Input: {len(texts)} text chunks")
print(f"   Output: {embeddings.shape[0]} vectors, each with {embeddings.shape[1]} dimensions")
print(f"   Total numbers stored: {embeddings.shape[0] * embeddings.shape[1]:,}")

# Step 3: Build our simulated vector database
# In production, this would be ChromaDB, Pinecone, Weaviate, etc.
# Here we use a simple dict to show the data structure clearly.
vector_store = []
for i, doc in enumerate(corpx_documents):
    vector_store.append({
        "id": i,
        "text": doc["text"],           # Original text (for returning to the user)
        "embedding": embeddings[i],     # The 384-dim vector (for similarity search)
        "metadata": {                   # Structured metadata (for filtering)
            "source": doc["source"],
            "category": doc["category"],
            "date": doc["date"],
        }
    })

print(f"\n✅ Vector store built with {len(vector_store)} entries")
print(f"\n   Example entry (first document):")
print(f"   - ID: {vector_store[0]['id']}")
print(f"   - Text: \"{vector_store[0]['text'][:80]}...\"")
print(f"   - Embedding shape: {vector_store[0]['embedding'].shape}")
print(f"   - Embedding (first 5 values): {vector_store[0]['embedding'][:5].round(4)}")
print(f"   - Metadata: {vector_store[0]['metadata']}")

print(f"\n💡 This is exactly what a real vector DB stores internally.")
print(f"   The embedding is what gets searched; the text and metadata are what get returned.")

## Step 3: How Similarity Search Works

Vector databases find the K nearest neighbors to your query vector. But "nearest" depends on which **distance metric** you use:

| Metric | What It Measures | Range | Used By |
|--------|-----------------|-------|---------|
| **Cosine Similarity** | Angle between vectors | -1 to 1 (1 = identical) | Most embedding models |
| **Euclidean (L2) Distance** | Straight-line distance | 0 to inf (0 = identical) | FAISS default |
| **Dot Product** | Magnitude-weighted alignment | -inf to inf (higher = more similar) | When vectors are normalized |

Let's implement all three and see where they agree — and where they don't.

In [ ]:
# ============================================================
# Implement all three similarity/distance metrics from scratch
# ============================================================

def compute_cosine_similarity(query_emb, doc_embeddings):
    """Cosine similarity: measures the angle between two vectors.
    cos(theta) = (A . B) / (||A|| * ||B||)
    Range: -1 (opposite) to 1 (identical direction).
    Ignores vector magnitude — only cares about direction."""
    # sklearn's cosine_similarity handles the normalization for us
    return cosine_similarity(query_emb.reshape(1, -1), doc_embeddings)[0]


def compute_euclidean_distance(query_emb, doc_embeddings):
    """Euclidean (L2) distance: straight-line distance in N-dimensional space.
    d = sqrt(sum((a_i - b_i)^2))
    Range: 0 (identical) to infinity (far apart).
    Sensitive to vector magnitude."""
    return euclidean_distances(query_emb.reshape(1, -1), doc_embeddings)[0]


def compute_dot_product(query_emb, doc_embeddings):
    """Dot product: sum of element-wise products.
    A . B = sum(a_i * b_i)
    Range: -inf to inf (higher = more similar).
    When vectors are normalized, dot product equals cosine similarity."""
    return np.dot(doc_embeddings, query_emb)


# ============================================================
# Run sample queries through all three metrics
# ============================================================

sample_queries = [
    "What is our revenue this quarter?",
    "How many people work at the company?",
    "Are clients happy with our services?",
    "What is the status of cloud migration?",
]

# Collect all embeddings from our vector store into a matrix
store_embeddings = np.array([entry["embedding"] for entry in vector_store])
store_texts = [entry["text"][:70] for entry in vector_store]

for query in sample_queries:
    q_emb = model.encode([query])[0]

    # Compute all three metrics
    cosine_scores = compute_cosine_similarity(q_emb, store_embeddings)
    euclidean_scores = compute_euclidean_distance(q_emb, store_embeddings)
    dot_scores = compute_dot_product(q_emb, store_embeddings)

    # Rank by each metric
    # For cosine and dot product: higher is better, so reverse sort
    # For euclidean: lower is better, so normal sort
    cosine_rank = np.argsort(cosine_scores)[::-1][:3]
    euclidean_rank = np.argsort(euclidean_scores)[:3]       # Lower distance = more similar
    dot_rank = np.argsort(dot_scores)[::-1][:3]

    print(f'\n🔍 Query: "{query}"')
    print(f"   {'-'*75}")
    print(f"   {'Rank':<6} {'Cosine Top-3':<28} {'Euclidean Top-3':<28} {'Dot Product Top-3'}")
    for r in range(3):
        # Show the document index for each metric's top result at this rank
        c_idx, e_idx, d_idx = cosine_rank[r], euclidean_rank[r], dot_rank[r]
        print(f"   {r+1:<6} Doc {c_idx:<2} (sim:{cosine_scores[c_idx]:.3f})  "
              f"Doc {e_idx:<2} (dist:{euclidean_scores[e_idx]:.3f})  "
              f"Doc {d_idx:<2} (dot:{dot_scores[d_idx]:.3f})")

    # Check if all three metrics agree on the top result
    if cosine_rank[0] == euclidean_rank[0] == dot_rank[0]:
        print(f"   ✅ All three metrics agree on the top result: Doc {cosine_rank[0]}")
    else:
        print(f"   ⚠️  Metrics DISAGREE on the top result!")

print(f"\n💡 Key insight: For normalized embeddings (like sentence-transformers), cosine")
print(f"   similarity and dot product produce identical rankings. Euclidean distance")
print(f"   usually agrees but can diverge when vectors have different magnitudes.")

In [ ]:
# ============================================================
# Visualization: Compare metric rankings for one query
# ============================================================

# Pick a query and visualize how each metric ranks ALL documents
query = "What is our revenue this quarter?"
q_emb = model.encode([query])[0]

cosine_scores = compute_cosine_similarity(q_emb, store_embeddings)
euclidean_scores = compute_euclidean_distance(q_emb, store_embeddings)
dot_scores = compute_dot_product(q_emb, store_embeddings)

# Normalize euclidean to 0-1 range for visual comparison (invert so higher = more similar)
euclidean_normalized = 1 - (euclidean_scores - euclidean_scores.min()) / (euclidean_scores.max() - euclidean_scores.min())
# Normalize dot product to 0-1 range
dot_normalized = (dot_scores - dot_scores.min()) / (dot_scores.max() - dot_scores.min())

# Create a grouped bar chart
fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(len(vector_store))
width = 0.25  # Width of each bar group

# Three bars per document, one for each metric
bars1 = ax.bar(x - width, cosine_scores, width, label='Cosine Similarity', color='#2E86C1', alpha=0.8)
bars2 = ax.bar(x, euclidean_normalized, width, label='Euclidean (normalized)', color='#E74C3C', alpha=0.8)
bars3 = ax.bar(x + width, dot_normalized, width, label='Dot Product (normalized)', color='#27AE60', alpha=0.8)

# Add category labels below each bar
categories = [entry['metadata']['category'] for entry in vector_store]
ax.set_xticks(x)
ax.set_xticklabels([f"Doc {i}\n({categories[i]})" for i in range(len(vector_store))],
                    fontsize=7, rotation=45, ha='right')

ax.set_ylabel('Similarity Score (normalized to 0-1)')
ax.set_title(f'Three Similarity Metrics Compared\nQuery: "{query}"',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print("💡 The three metrics generally agree on which documents are most relevant.")
print("   Minor ranking differences can occur — this is why metric choice matters")
print("   in production, especially for borderline results near the threshold.")

## Step 4: Deterministic vs Non-Deterministic Retrieval

This is a **critical concept** that trips up many engineers.

**SQL queries are deterministic:** the same `SELECT` statement on the same data returns the same rows, every time, forever.

**Vector search is non-deterministic in practice** because:

1. **Embedding model versions** — upgrading the model changes all vectors
2. **Approximate Nearest Neighbor (ANN) algorithms** — HNSW, IVF trade accuracy for speed
3. **Similarity scores are continuous** — a tiny threshold change flips results in/out
4. **Document updates change the vector space** — adding/removing docs shifts rankings

Let's demonstrate each of these with code.

In [ ]:
# ============================================================
# DEMO 1: Adding one new document shifts similarity rankings
# ============================================================
# This shows that the "answer" to the same query can change
# simply because new data was added to the index.

query = "What is our financial outlook?"
q_emb = model.encode([query])[0]

# --- BEFORE: Search against the original 18 documents ---
scores_before = cosine_similarity(q_emb.reshape(1, -1), store_embeddings)[0]
rank_before = np.argsort(scores_before)[::-1][:5]

print("🔍 DEMO 1: How adding ONE document changes rankings")
print(f'   Query: "{query}"')
print(f"\n   === BEFORE (18 documents in index) ===")
for i, idx in enumerate(rank_before[:5]):
    cat = vector_store[idx]['metadata']['category']
    print(f"   {i+1}. (sim: {scores_before[idx]:.4f}) [{cat}] {store_texts[idx]}")

# --- Add one highly relevant new document ---
new_doc_text = "The CFO projects Q4 revenue of $41M based on current pipeline and seasonal trends, with operating margins expected to hold steady."
new_doc_emb = model.encode([new_doc_text])

# Append to our embedding matrix
store_embeddings_after = np.vstack([store_embeddings, new_doc_emb])
store_texts_after = store_texts + [new_doc_text[:70]]

# --- AFTER: Search against 19 documents ---
scores_after = cosine_similarity(q_emb.reshape(1, -1), store_embeddings_after)[0]
rank_after = np.argsort(scores_after)[::-1][:5]

print(f"\n   === AFTER (19 documents — added CFO projection) ===")
for i, idx in enumerate(rank_after[:5]):
    text = store_texts_after[idx]
    is_new = " ← NEW" if idx == len(store_texts) else ""
    print(f"   {i+1}. (sim: {scores_after[idx]:.4f}) {text}{is_new}")

# Show which documents shifted position
print(f"\n   📊 Ranking changes:")
before_list = list(rank_before[:5])
after_list = list(rank_after[:5])
for idx in before_list:
    if idx not in after_list:
        print(f"   ✗ Doc {idx} was in top-5, now pushed out")
for idx in after_list:
    if idx == len(store_texts):
        print(f"   ✅ New document entered the top-5")

print(f"\n💡 Same query, different results — just because one document was added.")
print(f"   In production, documents are added/updated continuously.")
print(f"   Your RAG system's answers can drift over time even if the code doesn't change.")

In [ ]:
# ============================================================
# DEMO 2: Tiny threshold change includes/excludes a result
# ============================================================
# Similarity scores are continuous (e.g., 0.7483).
# A threshold of 0.75 vs 0.74 can mean a document is IN or OUT.

query = "employee satisfaction and engagement"
q_emb = model.encode([query])[0]
scores = cosine_similarity(q_emb.reshape(1, -1), store_embeddings)[0]

# Sort documents by score
sorted_idx = np.argsort(scores)[::-1]

# Define two thresholds that are very close together
threshold_high = 0.42
threshold_low = 0.38

print("🔍 DEMO 2: How a tiny threshold change flips results")
print(f'   Query: "{query}"')
print(f"\n   All documents ranked by similarity:")
print(f"   {'Rank':<6} {'Score':<10} {'Threshold 0.42':<18} {'Threshold 0.38':<18} Text")
print(f"   {'-'*6} {'-'*10} {'-'*18} {'-'*18} {'-'*50}")

# Count how many documents each threshold includes
count_high = 0
count_low = 0

for rank, idx in enumerate(sorted_idx[:12]):  # Show top 12
    score = scores[idx]
    # Determine if this document passes each threshold
    in_high = "✅ INCLUDED" if score >= threshold_high else "✗ excluded"
    in_low = "✅ INCLUDED" if score >= threshold_low else "✗ excluded"
    if score >= threshold_high:
        count_high += 1
    if score >= threshold_low:
        count_low += 1

    # Highlight the "flip zone" — documents between the two thresholds
    flip = " ← FLIP ZONE" if threshold_low <= score < threshold_high else ""
    print(f"   {rank+1:<6} {score:<10.4f} {in_high:<18} {in_low:<18} {store_texts[idx][:45]}...{flip}")

print(f"\n   📊 Summary:")
print(f"   Threshold 0.42 → {count_high} documents included")
print(f"   Threshold 0.38 → {count_low} documents included")
print(f"   Difference: {count_low - count_high} extra document(s) from a 0.04 threshold shift")
print(f"\n💡 In production, the 'right' threshold is hard to determine.")
print(f"   A change from 0.42 to 0.38 can mean the LLM sees different context and gives a different answer.")

In [ ]:
# ============================================================
# DEMO 3: Semantically similar queries return DIFFERENT top-3
# ============================================================
# Two ways of asking "the same question" can retrieve
# different documents because the embeddings differ.

query_a = "How much money did we make?"
query_b = "What were the financial results for the quarter?"

emb_a = model.encode([query_a])[0]
emb_b = model.encode([query_b])[0]

# Check how similar these two queries are to each other
query_similarity = cosine_similarity(
    emb_a.reshape(1, -1), emb_b.reshape(1, -1)
)[0][0]

# Get top-3 results for each query
scores_a = cosine_similarity(emb_a.reshape(1, -1), store_embeddings)[0]
scores_b = cosine_similarity(emb_b.reshape(1, -1), store_embeddings)[0]
top3_a = np.argsort(scores_a)[::-1][:3]
top3_b = np.argsort(scores_b)[::-1][:3]

print("🔍 DEMO 3: Same intent, different wording → different results")
print(f"\n   Query A: \"{query_a}\"")
print(f"   Query B: \"{query_b}\"")
print(f"   Similarity between the two queries: {query_similarity:.4f}")
print(f"   (These are clearly asking the same thing!)")

print(f"\n   Top-3 results for Query A:")
for i, idx in enumerate(top3_a):
    print(f"   {i+1}. (sim: {scores_a[idx]:.4f}) Doc {idx}: {store_texts[idx]}")

print(f"\n   Top-3 results for Query B:")
for i, idx in enumerate(top3_b):
    print(f"   {i+1}. (sim: {scores_b[idx]:.4f}) Doc {idx}: {store_texts[idx]}")

# Compare the result sets
overlap = set(top3_a) & set(top3_b)
only_a = set(top3_a) - set(top3_b)
only_b = set(top3_b) - set(top3_a)

print(f"\n   📊 Overlap analysis:")
print(f"   Shared in both top-3: {len(overlap)} document(s) — Docs {overlap if overlap else '{}'}")
print(f"   Only in Query A top-3: {only_a if only_a else '{}'}")
print(f"   Only in Query B top-3: {only_b if only_b else '{}'}")

print(f"\n💡 Even though these queries mean the same thing (sim: {query_similarity:.3f}),")
print(f"   the retrieved documents can differ. This means two users asking the 'same'")
print(f"   question in different words may get different answers from the LLM.")

In [ ]:
# ============================================================
# Visualization: Before/After comparison for Demo 1
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before: Top-5 rankings
top5_before_scores = [scores_before[idx] for idx in rank_before[:5]]
top5_before_labels = [f"Doc {idx}: {store_texts[idx][:40]}..." for idx in rank_before[:5]]

axes[0].barh(range(5), top5_before_scores, color='#2E86C1', edgecolor='white')
axes[0].set_yticks(range(5))
axes[0].set_yticklabels(top5_before_labels, fontsize=8)
axes[0].set_xlim(0, 0.8)
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_title('BEFORE: Top-5 (18 docs)', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

# After: Top-5 rankings (with new doc highlighted)
top5_after_scores = [scores_after[idx] for idx in rank_after[:5]]
top5_after_labels = []
top5_after_colors = []
for idx in rank_after[:5]:
    if idx == len(store_texts):  # The new document
        top5_after_labels.append(f"NEW: {new_doc_text[:40]}...")
        top5_after_colors.append('#E74C3C')  # Red for new
    else:
        top5_after_labels.append(f"Doc {idx}: {store_texts[idx][:40]}...")
        top5_after_colors.append('#2E86C1')

axes[1].barh(range(5), top5_after_scores, color=top5_after_colors, edgecolor='white')
axes[1].set_yticks(range(5))
axes[1].set_yticklabels(top5_after_labels, fontsize=8)
axes[1].set_xlim(0, 0.8)
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_title('AFTER: Top-5 (19 docs — new doc in red)', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

plt.suptitle('Adding One Document Changes Rankings',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 The new document (red) pushed an existing result out of the top-5.")
print("   In a RAG system, this means the LLM now sees different context.")

## Step 5: The Wrong Context Problem

This is where vector databases silently break your RAG system.

The LLM doesn't know if the retrieved context is correct, relevant, or up-to-date. It **trusts whatever you put in the prompt** and generates a confident-sounding answer regardless. If the vector DB returns the wrong chunks, the LLM produces a wrong answer — and the user has no way to tell.

Let's see three failure modes.

In [ ]:
# ============================================================
# Helper function: simulate a RAG retrieval + prompt assembly
# ============================================================

def simulate_retrieval(query, store_embeddings, vector_store, model, top_k=3):
    """Retrieve top-K documents and assemble a RAG prompt.
    Returns the results, scores, and assembled prompt."""
    q_emb = model.encode([query])[0]
    scores = cosine_similarity(q_emb.reshape(1, -1), store_embeddings)[0]
    ranked = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in ranked:
        results.append({
            "idx": idx,
            "text": vector_store[idx]["text"],
            "score": scores[idx],
            "metadata": vector_store[idx]["metadata"],
        })

    # Assemble the RAG prompt that would be sent to the LLM
    context = "\n".join([f"- {r['text']}" for r in results])
    prompt = f"""SYSTEM: You are a senior analyst at CorpX. Answer ONLY based on the provided context.

CONTEXT:
{context}

USER QUESTION: {query}

ANSWER:"""

    return results, scores, prompt

In [ ]:
# ============================================================
# SCENARIO 1: Ambiguous query retrieves a random mix
# ============================================================
# When the user asks something vague, the vector DB returns
# documents from multiple unrelated categories.

vague_query = "How are we doing?"
results, scores, prompt = simulate_retrieval(
    vague_query, store_embeddings, vector_store, model, top_k=5
)

print("="*80)
print("SCENARIO 1: Ambiguous Query → Random Mix of Context")
print("="*80)
print(f'\n🔍 Query: "{vague_query}"')
print(f"\n   Retrieved documents (top-5):")

# Track which categories appear — a good retrieval would be focused
retrieved_categories = []
for i, r in enumerate(results):
    cat = r['metadata']['category']
    retrieved_categories.append(cat)
    print(f"   {i+1}. (sim: {r['score']:.4f}) [{cat}] {r['text'][:75]}...")

unique_cats = set(retrieved_categories)
print(f"\n   📊 Categories retrieved: {unique_cats}")
print(f"   Number of different categories: {len(unique_cats)}")

print(f"\n   ⚠️  THE PROBLEM:")
print(f"   The query 'How are we doing?' is too vague. The vector DB returned")
print(f"   documents from {len(unique_cats)} different categories because it matches")
print(f"   weakly with everything and strongly with nothing.")
print(f"\n   The LLM would receive this mixed context and produce a rambling answer")
print(f"   that touches on many topics without depth on any of them.")

print(f"\n   📋 Assembled prompt (what the LLM would see):")
print(f"   {'-'*60}")
# Only show first few lines to keep output manageable
for line in prompt.split('\n')[:10]:
    print(f"   {line}")
print(f"   ...")
print(f"   {'-'*60}")

In [ ]:
# ============================================================
# SCENARIO 2: Outdated context — stale documents rank high
# ============================================================
# When old and new documents coexist in the index, the vector DB
# may return outdated info because it only measures text similarity,
# not freshness.

# Our vector store has both Q2 and Q3 financial data.
# Let's query for "current revenue" and see what happens.

freshness_query = "What is our current revenue?"
results, scores, prompt = simulate_retrieval(
    freshness_query, store_embeddings, vector_store, model, top_k=4
)

print("="*80)
print("SCENARIO 2: Outdated Context — Stale Documents in Results")
print("="*80)
print(f'\n🔍 Query: "{freshness_query}"')
print(f"\n   Retrieved documents:")

for i, r in enumerate(results):
    date = r['metadata']['date']
    source = r['metadata']['source']
    # Flag old documents
    freshness = "⚠️  OLD" if "Q2" in source else "✅ CURRENT"
    print(f"   {i+1}. (sim: {r['score']:.4f}) [{date}] {freshness} {r['text'][:70]}...")

# Check if any outdated docs made it into the results
outdated = [r for r in results if 'Q2' in r['metadata']['source']]
print(f"\n   ⚠️  THE PROBLEM:")
if outdated:
    print(f"   {len(outdated)} out of {len(results)} retrieved documents are from Q2 (outdated).")
    print(f"   The user asked for 'current' revenue, but the vector DB doesn't understand")
    print(f"   time — it just matches semantic similarity. Old and new revenue docs")
    print(f"   both mention 'revenue' so they both rank highly.")
    print(f"\n   The LLM might cite the Q2 number ($33.7M) instead of Q3 ($38.4M),")
    print(f"   or worse, average them together in a confused answer.")
else:
    print(f"   In this case, only current docs were retrieved — but with a larger corpus")
    print(f"   containing years of data, outdated results become increasingly likely.")

In [ ]:
# ============================================================
# SCENARIO 3: Chunk boundary problem — split mid-sentence
# ============================================================
# Real documents get split into chunks. If the split happens
# in the wrong place, the important information lands in the
# OTHER chunk that doesn't get retrieved.

# Simulate a document that was badly chunked
original_paragraph = (
    "CorpX's AI practice revenue reached $5.9 million in Q3, "
    "representing 111% year-over-year growth. This growth was primarily driven "
    "by three large federal contracts for AI-powered document processing "
    "systems. The pipeline for Q4 includes an additional $8M in qualified "
    "opportunities from both federal and commercial clients."
)

# BAD chunking: split right in the middle, separating the numbers from context
chunk_a = "CorpX's AI practice revenue reached $5.9 million in Q3, representing 111% year-over-year growth. This growth was primarily"
chunk_b = "driven by three large federal contracts for AI-powered document processing systems. The pipeline for Q4 includes an additional $8M in qualified opportunities from both federal and commercial clients."

# Embed both chunks
chunk_a_emb = model.encode([chunk_a])
chunk_b_emb = model.encode([chunk_b])

# Query that should find the AI revenue info
chunk_query = "What is driving AI revenue growth?"
chunk_q_emb = model.encode([chunk_query])

# Check similarity to each chunk
sim_to_a = cosine_similarity(chunk_q_emb, chunk_a_emb)[0][0]
sim_to_b = cosine_similarity(chunk_q_emb, chunk_b_emb)[0][0]

print("="*80)
print("SCENARIO 3: Chunk Boundary Problem")
print("="*80)

print(f"\n   Original paragraph (before chunking):")
print(f"   {'-'*60}")
# Wrap text for display
for line in textwrap.wrap(original_paragraph, width=70):
    print(f"   {line}")
print(f"   {'-'*60}")

print(f"\n   After bad chunking (split mid-sentence):")
print(f"   Chunk A: \"{chunk_a}\"")
print(f"   Chunk B: \"{chunk_b}\"")

print(f'\n🔍 Query: "{chunk_query}"')
print(f"   Similarity to Chunk A (has numbers, not the drivers): {sim_to_a:.4f}")
print(f"   Similarity to Chunk B (has the drivers, not the numbers): {sim_to_b:.4f}")

# Which chunk would be retrieved?
if sim_to_b > sim_to_a:
    print(f"\n   ⚠️  THE PROBLEM:")
    print(f"   Chunk B scores higher because it contains the 'driving' factors.")
    print(f"   But Chunk B doesn't have the revenue figure ($5.9M) or the growth rate (111%).")
    print(f"   The LLM would describe the drivers but couldn't cite the actual numbers.")
else:
    print(f"\n   ⚠️  THE PROBLEM:")
    print(f"   Chunk A scores higher because it mentions 'revenue' and 'growth'.")
    print(f"   But Chunk A doesn't explain WHAT drove the growth (federal contracts).")
    print(f"   The LLM would cite numbers but couldn't explain the underlying cause.")

print(f"\n   Either way, the LLM only sees HALF the picture and generates an incomplete answer.")
print(f"   This is why chunking strategy matters — see notebook 03 for details.")

In [ ]:
# ============================================================
# Visualization: "Close But Wrong" — similarity scores
# for all three failure scenarios
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Scenario 1: Ambiguous query ---
vague_q_emb = model.encode(["How are we doing?"])[0]
vague_scores = cosine_similarity(vague_q_emb.reshape(1, -1), store_embeddings)[0]
vague_sorted = np.argsort(vague_scores)[::-1][:8]

colors_s1 = []
for idx in vague_sorted:
    cat = vector_store[idx]['metadata']['category']
    # Color by category to show the random mix
    cat_colors = {'financial': '#2E86C1', 'hr': '#1E8449', 'client': '#D35400',
                  'tech': '#8E44AD', 'strategy': '#F39C12'}
    colors_s1.append(cat_colors.get(cat, '#999999'))

axes[0].barh(range(8), [vague_scores[i] for i in vague_sorted],
             color=colors_s1, edgecolor='white')
axes[0].set_yticks(range(8))
axes[0].set_yticklabels(
    [f"[{vector_store[i]['metadata']['category']}] {store_texts[i][:30]}..."
     for i in vague_sorted], fontsize=7)
axes[0].set_title('Scenario 1: Ambiguous Query\n"How are we doing?"',
                   fontsize=10, fontweight='bold')
axes[0].set_xlabel('Cosine Similarity')
axes[0].invert_yaxis()
# Note: all scores are close together — no clear winner
axes[0].axvline(x=np.mean(vague_scores[vague_sorted[:8]]), color='red',
                linestyle='--', alpha=0.5, label='mean')
axes[0].legend(fontsize=8)

# --- Scenario 2: Outdated context ---
fresh_q_emb = model.encode(["What is our current revenue?"])[0]
fresh_scores = cosine_similarity(fresh_q_emb.reshape(1, -1), store_embeddings)[0]
fresh_sorted = np.argsort(fresh_scores)[::-1][:8]

colors_s2 = []
for idx in fresh_sorted:
    source = vector_store[idx]['metadata']['source']
    # Highlight Q2 (outdated) docs in red
    colors_s2.append('#E74C3C' if 'Q2' in source else '#2E86C1')

axes[1].barh(range(8), [fresh_scores[i] for i in fresh_sorted],
             color=colors_s2, edgecolor='white')
axes[1].set_yticks(range(8))
axes[1].set_yticklabels(
    [f"[{vector_store[i]['metadata']['date']}] {store_texts[i][:30]}..."
     for i in fresh_sorted], fontsize=7)
axes[1].set_title('Scenario 2: Outdated Context\n"Current revenue?" (red=old)',
                   fontsize=10, fontweight='bold')
axes[1].set_xlabel('Cosine Similarity')
axes[1].invert_yaxis()

# --- Scenario 3: Chunk boundary ---
chunk_labels = ['Chunk A\n(has numbers)', 'Chunk B\n(has drivers)']
chunk_sims = [sim_to_a, sim_to_b]
chunk_colors = ['#F39C12', '#8E44AD']
axes[2].barh(range(2), chunk_sims, color=chunk_colors, edgecolor='white', height=0.5)
axes[2].set_yticks(range(2))
axes[2].set_yticklabels(chunk_labels, fontsize=9)
axes[2].set_title('Scenario 3: Chunk Boundary\n"What drives AI revenue growth?"',
                   fontsize=10, fontweight='bold')
axes[2].set_xlabel('Cosine Similarity')
axes[2].invert_yaxis()
# Annotate which chunk gets retrieved
winner = 0 if sim_to_a > sim_to_b else 1
axes[2].annotate('← RETRIEVED', xy=(chunk_sims[winner], winner),
                  fontsize=9, fontweight='bold', color='red',
                  xytext=(0.05, 0), textcoords='offset points', va='center')

plt.suptitle('The Wrong Context Problem: Three Failure Modes',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 In all three scenarios, the vector DB returned results that LOOK relevant")
print("   (high similarity scores) but would cause the LLM to produce wrong answers.")
print("   The LLM has no way to detect this — it trusts whatever context it receives.")

## Step 6: Improving Retrieval Quality

Now that we've seen what can go wrong, let's fix it. Here are four practical techniques to improve retrieval quality:

1. **Hybrid Search** — combine keyword matching with semantic similarity
2. **Metadata Filtering** — filter by category/date BEFORE similarity search
3. **Reranking** — retrieve more, then rerank to find the best
4. **Query Transformation** — rephrase vague queries into specific ones

In [ ]:
# ============================================================
# TECHNIQUE 1: Hybrid Search (keyword + semantic)
# ============================================================
# Combine traditional keyword matching (BM25-style) with
# semantic similarity. This catches results that are lexically
# relevant even if semantically distant, and vice versa.

def keyword_score(query, text):
    """Simple keyword overlap score (simulating BM25).
    Counts what fraction of query terms appear in the document."""
    query_terms = set(query.lower().split())
    text_terms = set(text.lower().split())
    # What fraction of query terms appear in the document?
    if not query_terms:
        return 0.0
    overlap = query_terms & text_terms
    return len(overlap) / len(query_terms)


def hybrid_search(query, vector_store, store_embeddings, model,
                  top_k=5, semantic_weight=0.7, keyword_weight=0.3):
    """Combine semantic similarity with keyword matching.
    The weights control the balance between the two signals."""
    # Semantic scores (cosine similarity)
    q_emb = model.encode([query])[0]
    semantic_scores = cosine_similarity(q_emb.reshape(1, -1), store_embeddings)[0]

    # Keyword scores
    kw_scores = np.array([
        keyword_score(query, entry["text"])
        for entry in vector_store
    ])

    # Weighted combination
    combined = (semantic_weight * semantic_scores) + (keyword_weight * kw_scores)

    # Rank by combined score
    ranked = np.argsort(combined)[::-1][:top_k]

    return ranked, semantic_scores, kw_scores, combined


# Compare pure semantic vs hybrid search on the ambiguous query from Scenario 1
query = "Q3 revenue growth performance"

# Pure semantic
q_emb = model.encode([query])[0]
semantic_only = cosine_similarity(q_emb.reshape(1, -1), store_embeddings)[0]
semantic_rank = np.argsort(semantic_only)[::-1][:5]

# Hybrid
hybrid_rank, sem_scores, kw_scores, combined_scores = hybrid_search(
    query, vector_store, store_embeddings, model, top_k=5
)

print("🔍 TECHNIQUE 1: Hybrid Search")
print(f'   Query: "{query}"')
print(f"\n   --- Pure Semantic Search ---")
for i, idx in enumerate(semantic_rank):
    cat = vector_store[idx]['metadata']['category']
    print(f"   {i+1}. (sem: {semantic_only[idx]:.4f}) [{cat}] {store_texts[idx][:60]}")

print(f"\n   --- Hybrid Search (70% semantic + 30% keyword) ---")
for i, idx in enumerate(hybrid_rank):
    cat = vector_store[idx]['metadata']['category']
    print(f"   {i+1}. (combined: {combined_scores[idx]:.4f} | sem: {sem_scores[idx]:.4f} | kw: {kw_scores[idx]:.4f}) [{cat}] {store_texts[idx][:50]}")

print(f"\n💡 Hybrid search boosts documents that match BOTH semantically and lexically.")
print(f"   Documents containing the actual words 'Q3', 'revenue', 'growth' get a bonus.")

In [ ]:
# ============================================================
# TECHNIQUE 2: Metadata Filtering
# ============================================================
# Filter by category or date BEFORE running similarity search.
# This directly fixes the "outdated context" problem from Scenario 2.

def filtered_search(query, vector_store, store_embeddings, model,
                    top_k=3, category=None, min_date=None):
    """Search with metadata pre-filtering.
    First narrow down the candidate set, then rank by similarity."""
    q_emb = model.encode([query])[0]

    # Pre-filter: only consider documents matching the metadata criteria
    candidates = []
    for i, entry in enumerate(vector_store):
        # Apply category filter if specified
        if category and entry["metadata"]["category"] != category:
            continue
        # Apply date filter if specified
        if min_date and entry["metadata"]["date"] < min_date:
            continue
        candidates.append(i)

    if not candidates:
        print("   No documents match the filters!")
        return []

    # Now compute similarity only on the filtered candidates
    candidate_embeddings = store_embeddings[candidates]
    scores = cosine_similarity(q_emb.reshape(1, -1), candidate_embeddings)[0]

    # Rank filtered candidates by similarity
    ranked = np.argsort(scores)[::-1][:top_k]

    results = []
    for r in ranked:
        orig_idx = candidates[r]
        results.append((orig_idx, scores[r]))

    return results


# Fix Scenario 2: filter by date to exclude outdated docs
query = "What is our current revenue?"

print("🔍 TECHNIQUE 2: Metadata Filtering")
print(f'   Query: "{query}"')

# Without filtering (the broken version)
print(f"\n   --- Without filtering (all dates) ---")
results_unfiltered = simulate_retrieval(query, store_embeddings, vector_store, model, top_k=3)
for i, r in enumerate(results_unfiltered[0]):
    date = r['metadata']['date']
    freshness = "⚠️  OLD" if '2024-07' in date else "✅"
    print(f"   {i+1}. (sim: {r['score']:.4f}) [{date}] {freshness} {r['text'][:60]}...")

# With date filtering — only Q3 data (October 2024+)
print(f"\n   --- With date filter (min_date='2024-10-01') ---")
results_filtered = filtered_search(
    query, vector_store, store_embeddings, model,
    top_k=3, min_date='2024-10-01'
)
for i, (idx, score) in enumerate(results_filtered):
    date = vector_store[idx]['metadata']['date']
    print(f"   {i+1}. (sim: {score:.4f}) [{date}] ✅ {vector_store[idx]['text'][:60]}...")

print(f"\n💡 By filtering on date metadata, we excluded the stale Q2 documents.")
print(f"   The LLM now only sees current data and will give an accurate answer.")
print(f"   This is why metadata design is critical in production vector DBs.")

In [ ]:
# ============================================================
# TECHNIQUE 3: Reranking (retrieve top-20, rerank to top-5)
# ============================================================
# The idea: cast a wide net with similarity search (cheap/fast),
# then use a more sophisticated scoring function to rerank.
# In production, this would be a cross-encoder model.
# Here we simulate with a relevance heuristic.

def rerank_by_query_coverage(query, candidates, vector_store):
    """Rerank by how many query concepts appear in the document.
    This simulates what a cross-encoder does — deeper text analysis."""
    query_lower = query.lower()
    scored = []
    for idx, sim_score in candidates:
        text = vector_store[idx]["text"].lower()
        # Score: count how many meaningful query words appear in the text
        # (skip stop words)
        stop_words = {'what', 'is', 'our', 'the', 'a', 'an', 'how', 'are', 'we', 'and', 'of', 'in', 'for', 'to'}
        query_words = [w for w in query_lower.split() if w not in stop_words]
        coverage = sum(1 for w in query_words if w in text) / max(len(query_words), 1)
        # Combined score: original similarity + coverage bonus
        rerank_score = 0.6 * sim_score + 0.4 * coverage
        scored.append((idx, sim_score, coverage, rerank_score))
    # Sort by reranked score
    scored.sort(key=lambda x: x[3], reverse=True)
    return scored


query = "AI revenue growth and federal contracts"
q_emb = model.encode([query])[0]
all_scores = cosine_similarity(q_emb.reshape(1, -1), store_embeddings)[0]

# Step 1: Retrieve top-10 (wide net)
top10_idx = np.argsort(all_scores)[::-1][:10]
candidates = [(idx, all_scores[idx]) for idx in top10_idx]

# Step 2: Rerank to find the best 5
reranked = rerank_by_query_coverage(query, candidates, vector_store)

print("🔍 TECHNIQUE 3: Reranking")
print(f'   Query: "{query}"')

print(f"\n   --- Initial retrieval (top-10 by cosine similarity) ---")
for i, (idx, sim) in enumerate(candidates[:5]):
    cat = vector_store[idx]['metadata']['category']
    print(f"   {i+1}. (sim: {sim:.4f}) [{cat}] {store_texts[idx][:55]}")
print(f"   ... + 5 more")

print(f"\n   --- After reranking (top-5 by combined score) ---")
for i, (idx, sim, coverage, rerank) in enumerate(reranked[:5]):
    cat = vector_store[idx]['metadata']['category']
    print(f"   {i+1}. (rerank: {rerank:.4f} | sim: {sim:.4f} | coverage: {coverage:.2f}) [{cat}] {store_texts[idx][:50]}")

# Check if order changed
initial_order = [c[0] for c in candidates[:5]]
reranked_order = [r[0] for r in reranked[:5]]
if initial_order != reranked_order:
    print(f"\n   ✅ Reranking changed the order! Some documents moved up because")
    print(f"      they have better concept coverage for this specific query.")
else:
    print(f"\n   Reranking kept the same order — the initial retrieval was already good.")

print(f"\n💡 In production, use a cross-encoder model (e.g., ms-marco-MiniLM) for reranking.")
print(f"   Cross-encoders analyze query+document pairs jointly — much more accurate")
print(f"   than cosine similarity, but too slow to run on the full corpus.")

In [ ]:
# ============================================================
# TECHNIQUE 4: Query Transformation
# ============================================================
# Rephrase vague queries into specific ones before searching.
# In production, you'd use an LLM to do this transformation.
# Here we show the effect manually.

# The vague query from Scenario 1
vague_query = "How are we doing?"

# Transformed into specific sub-queries
# (In production, an LLM would generate these)
specific_queries = [
    "What is our Q3 revenue and financial performance?",
    "What are employee headcount and attrition trends?",
    "How satisfied are our clients this quarter?",
]

print("🔍 TECHNIQUE 4: Query Transformation")
print(f'   Original vague query: "{vague_query}"')
print(f"\n   Transformed into {len(specific_queries)} specific sub-queries:")

all_retrieved = []  # Collect all results across sub-queries

for sq in specific_queries:
    print(f'\n   Sub-query: "{sq}"')
    results, _, _ = simulate_retrieval(sq, store_embeddings, vector_store, model, top_k=2)
    for r in results:
        cat = r['metadata']['category']
        print(f"     → (sim: {r['score']:.4f}) [{cat}] {r['text'][:60]}...")
        all_retrieved.append(r)

# Compare: original vague query results vs transformed results
print(f"\n   {'='*60}")
print(f"   COMPARISON:")

# Vague query categories
vague_results, _, _ = simulate_retrieval(vague_query, store_embeddings, vector_store, model, top_k=5)
vague_cats = set(r['metadata']['category'] for r in vague_results)
transformed_cats = set(r['metadata']['category'] for r in all_retrieved)

print(f"   Vague query retrieved from categories: {vague_cats}")
print(f"   Transformed queries retrieved from: {transformed_cats}")

print(f"\n💡 Query transformation turns one vague question into multiple focused ones.")
print(f"   Each sub-query retrieves the BEST documents for its specific topic.")
print(f"   The LLM gets comprehensive, focused context instead of a random mix.")

In [ ]:
# ============================================================
# Visualization: Before/After for the ambiguous query
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: Vague query results (from Scenario 1) ---
vague_results_data, _, _ = simulate_retrieval(
    "How are we doing?", store_embeddings, vector_store, model, top_k=6
)

vague_labels = []
vague_scores_list = []
vague_colors = []
cat_color_map = {'financial': '#2E86C1', 'hr': '#1E8449', 'client': '#D35400',
                 'tech': '#8E44AD', 'strategy': '#F39C12'}

for r in vague_results_data:
    cat = r['metadata']['category']
    vague_labels.append(f"[{cat}] {r['text'][:40]}...")
    vague_scores_list.append(r['score'])
    vague_colors.append(cat_color_map.get(cat, '#999'))

axes[0].barh(range(len(vague_labels)), vague_scores_list,
             color=vague_colors, edgecolor='white')
axes[0].set_yticks(range(len(vague_labels)))
axes[0].set_yticklabels(vague_labels, fontsize=7)
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_title('BEFORE: Vague Query\n"How are we doing?"\n(random category mix)',
                   fontsize=10, fontweight='bold')
axes[0].invert_yaxis()

# --- Right: Transformed query results (focused) ---
transformed_labels = []
transformed_scores_list = []
transformed_colors = []

for r in all_retrieved:
    cat = r['metadata']['category']
    transformed_labels.append(f"[{cat}] {r['text'][:40]}...")
    transformed_scores_list.append(r['score'])
    transformed_colors.append(cat_color_map.get(cat, '#999'))

axes[1].barh(range(len(transformed_labels)), transformed_scores_list,
             color=transformed_colors, edgecolor='white')
axes[1].set_yticks(range(len(transformed_labels)))
axes[1].set_yticklabels(transformed_labels, fontsize=7)
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_title('AFTER: Transformed Sub-Queries\n(focused, higher scores)',
                   fontsize=10, fontweight='bold')
axes[1].invert_yaxis()

plt.suptitle('Query Transformation: Vague → Specific',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Notice the right panel has higher similarity scores AND more focused categories.")
print("   Each sub-query retrieved the BEST match for its topic, not a random mix.")

## Step 7: Vector DB Landscape

There are many vector database options, each with different trade-offs. Here's a quick comparison:

| Vector DB | Type | Best For | ANN Algorithm | Managed? | Notable Feature |
|-----------|------|----------|---------------|----------|----------------|
| **ChromaDB** | Embedded | Prototyping, small datasets | HNSW | No (local) | Simple Python API, great for notebooks |
| **Pinecone** | Cloud-native | Production SaaS apps | Proprietary | Yes (fully) | Zero-ops, auto-scaling, namespaces |
| **Weaviate** | Self-hosted / Cloud | Hybrid search, multi-modal | HNSW | Both | Built-in vectorization modules |
| **Azure AI Search** | Cloud-native | Enterprise / Azure ecosystem | HNSW | Yes (fully) | Integrated with Azure OpenAI, RBAC |
| **pgvector** | Postgres extension | Teams already using Postgres | IVFFlat / HNSW | Both | No new infra — just an extension |
| **OpenSearch** | Self-hosted / Cloud | Log analytics + vector search | HNSW / NMSLIB | Both | Combines full-text and vector search |

**How to choose:**
- **Just learning / prototyping?** → ChromaDB (no setup, runs in a notebook)
- **Building a SaaS product?** → Pinecone or Weaviate Cloud
- **Enterprise with Azure?** → Azure AI Search (native integration with Azure OpenAI)
- **Already using Postgres?** → pgvector (add vector search to your existing DB)
- **Need full-text + vector?** → OpenSearch or Weaviate (hybrid search built-in)

## Key Takeaways

1. **Vector databases store meaning, not rows.** Traditional DBs query by exact match (`WHERE revenue > 100`). Vector DBs query by similarity ("find documents like this"). Both have their place — many production systems use both.

2. **Similarity search is non-deterministic in practice.** Adding documents, changing thresholds, or rephrasing queries can all change the results. Build your RAG system knowing that retrieval is approximate, not exact.

3. **The LLM trusts whatever context it receives.** If the vector DB returns wrong, outdated, or incomplete chunks, the LLM will generate a confident-sounding wrong answer. Retrieval quality is the bottleneck of most RAG systems.

4. **Metric choice matters at the margins.** Cosine similarity, Euclidean distance, and dot product usually agree on the top results, but can diverge for borderline documents — which are exactly the ones that get included or excluded by your threshold.

5. **Four techniques to improve retrieval:** Hybrid search (keyword + semantic), metadata filtering (pre-filter by date/category), reranking (retrieve many, select few), and query transformation (rephrase vague queries into specific ones).

6. **Design your metadata schema carefully.** The ability to filter by date, category, source, and other structured fields is what separates a good vector DB deployment from a bad one.

---

### Next Steps
- **[01-semantic-search-visualized.ipynb](01-semantic-search-visualized.ipynb)** — See semantic search fundamentals in action
- **[02-build-a-rag-pipeline.ipynb](02-build-a-rag-pipeline.ipynb)** — Build a full RAG system with a real vector database and LLM
- **[03-chunking-strategies.ipynb](03-chunking-strategies.ipynb)** — How chunking affects what gets embedded and retrieved
- **[04-embeddings-explorer.ipynb](04-embeddings-explorer.ipynb)** — Interactive 3D visualization of embedding space